# Next-Word Prediction / Autocomplete — Multi-Approach Comparison
### English & Arabic | Kaggle Notebook

This notebook builds and compares **three approaches** to next-word prediction (autocomplete):

1. **N-gram statistical language model** (with Laplace smoothing + stupid backoff)
2. **LSTM/RNN neural language model** (Keras, trained from scratch on the corpus)
3. **Pretrained Transformer** (GPT-2 for English, AraGPT2 for Arabic — zero-shot, no training needed)

Both **English** and **Arabic** are supported throughout, with Arabic-specific normalization
(diacritic removal, alef/teh-marbuta unification, elongation collapse) applied before tokenization.

**Real data:** this notebook trains on **real Wikipedia articles**, streamed live via the Hugging
Face `datasets` library (`wikimedia/wikipedia`, English + Arabic configs) — no synthetic text. A
small number of articles is pulled per language (configurable) to keep training time reasonable on
Kaggle; increase `N_ARTICLES` / `MAX_CHARS` for a larger, higher-quality corpus.

**Design goals (production-style, matching your Session 06 patterns):**
- Environment-aware data loading: local Kaggle dataset path (if you set one) → real streamed Wikipedia data → tiny built-in sample text, in that priority order, so the notebook *always* completes even if streaming fails (e.g. internet off)
- No blocking `input()` calls in the main run path — Run All will complete without hanging
- Every section ends with a runnable demo so you can sanity-check as you go
- Models, tokenizers, and vocabularies are saved to `/kaggle/working/` at the end

**How to use your own corpus instead:** attach a Kaggle dataset and set `EN_CORPUS_PATH` /
`AR_CORPUS_PATH` in the Data Loading section below to the `.txt` file path — that takes priority
over the streamed Wikipedia data.


## 0. Setup & Installs

In [1]:
# Kaggle: internet must be ON (Settings panel, right sidebar) for pip installs + pretrained transformer
# downloads + streaming the real Wikipedia dataset below.
!pip install -q transformers sentencepiece nltk arabic-reshaper python-bidi datasets --upgrade
print("Install step complete.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 39.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 30.9 MB/s eta 0:00:00
Install step complete.


In [2]:
import os
import re
import json
import time
import pickle
import random
import warnings
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    GPT2LMHeadModel, GPT2Tokenizer
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("TensorFlow:", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))
print("Torch device:", DEVICE)


2026-06-21 15:34:55.227358: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782056095.405745      59 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782056095.454467      59 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782056095.897431      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782056095.897474      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782056095.897477      59 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0 | GPUs: 0
Torch device: cpu


2026-06-21 15:35:22.226943: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 1. Data Loading — Real Wikipedia Articles (English & Arabic)

Streamed live from the [`wikimedia/wikipedia`](https://huggingface.co/datasets/wikimedia/wikipedia)
dataset on Hugging Face — actual encyclopedia text, not synthetic sentences. Streaming mode pulls
only as many articles as needed, so nothing huge gets downloaded.

If you'd rather use your own attached Kaggle dataset, set `EN_CORPUS_PATH` / `AR_CORPUS_PATH` to a
`.txt` file path — that takes priority over the streamed data.


In [3]:
# --- Configuration ---
EN_CORPUS_PATH = None   # e.g. "/kaggle/input/my-english-corpus/corpus.txt" — overrides streaming if set
AR_CORPUS_PATH = None   # e.g. "/kaggle/input/my-arabic-corpus/corpus.txt"  — overrides streaming if set

N_ARTICLES = 300        # number of real Wikipedia articles to pull per language
MAX_CHARS = 600_000     # safety cap on total characters per language (keeps training time reasonable)

# Wikipedia dump configs to try, in order (newer dumps occasionally get superseded on the Hub)
WIKI_CONFIGS_EN = ["20231101.en", "20220301.en"]
WIKI_CONFIGS_AR = ["20231101.ar", "20220301.ar"]

# Tiny built-in fallback (only used if BOTH a local path is absent AND streaming fails, e.g. no internet)
SAMPLE_EN_CORPUS = """
Artificial intelligence is transforming the way people live and work around the world.
Machine learning models can now understand natural language and generate human like text.
Deep learning has become the backbone of modern natural language processing systems.
Natural language processing allows computers to read understand and generate human language.
Word embeddings represent words as dense vectors that capture semantic meaning and relationships.
""".strip()

SAMPLE_AR_CORPUS = """
الذكاء الاصطناعي يغير طريقة عيش وعمل الناس في جميع أنحاء العالم.
تستطيع نماذج تعلم الآلة الآن فهم اللغة الطبيعية وتوليد نصوص شبيهة بالنصوص البشرية.
أصبح التعلم العميق العمود الفقري لأنظمة معالجة اللغة الطبيعية الحديثة.
تسمح معالجة اللغة الطبيعية للحواسيب بقراءة وفهم وتوليد اللغة البشرية.
تمثل تضمينات الكلمات الكلمات كمتجهات كثيفة تلتقط المعنى الدلالي والعلاقات بين الكلمات.
""".strip()


def stream_real_wikipedia(configs, label, n_articles=N_ARTICLES, max_chars=MAX_CHARS):
    """Pull real Wikipedia articles via streaming (no full dataset download)."""
    from datasets import load_dataset
    for cfg in configs:
        try:
            print(f"[{label}] Streaming real Wikipedia data (config='{cfg}')...")
            ds = load_dataset("wikimedia/wikipedia", cfg, split="train", streaming=True)
            texts, total_chars = [], 0
            for i, example in enumerate(ds):
                if i >= n_articles or total_chars >= max_chars:
                    break
                t = example["text"]
                texts.append(t)
                total_chars += len(t)
            text = "\n".join(texts)
            print(f"[{label}] Pulled {len(texts)} real articles, {len(text):,} characters "
                  f"(config='{cfg}').")
            if text.strip():
                return text
        except Exception as e:
            print(f"[{label}] Config '{cfg}' failed ({e}); trying next option...")
    return None


def get_corpus(path, configs, label, fallback):
    if path and os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        print(f"[{label}] Loaded local file: {len(text):,} characters from {path}")
        return text
    real_text = stream_real_wikipedia(configs, label)
    if real_text:
        return real_text
    print(f"[{label}] Streaming failed (likely no internet) — using tiny built-in fallback corpus. "
          f"Enable internet in Kaggle settings to use real data.")
    return fallback


en_text = get_corpus(EN_CORPUS_PATH, WIKI_CONFIGS_EN, "English", SAMPLE_EN_CORPUS)
ar_text = get_corpus(AR_CORPUS_PATH, WIKI_CONFIGS_AR, "Arabic", SAMPLE_AR_CORPUS)


[English] Streaming real Wikipedia data (config='20231101.en')...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

[English] Pulled 18 real articles, 601,947 characters (config='20231101.en').
[Arabic] Streaming real Wikipedia data (config='20231101.ar')...
[Arabic] Pulled 27 real articles, 633,062 characters (config='20231101.ar').


## 2. Preprocessing

English: lowercase + NLTK word tokenization.
Arabic: diacritic removal, alef/teh-marbuta/yaa normalization, elongation (tatweel) collapse,
then whitespace tokenization (same normalization strategy used in the Arabic sentiment notebook).


In [4]:
# ---------- English preprocessing ----------
def preprocess_english(text):
    text = text.lower()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    tokenized = [word_tokenize(s) for s in sentences if s.strip()]
    tokenized = [[w for w in s if re.match(r"^[a-z']+$", w)] for s in tokenized]
    return [s for s in tokenized if len(s) >= 3]

# ---------- Arabic preprocessing ----------
ARABIC_DIACRITICS = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]')

def normalize_arabic(text):
    text = ARABIC_DIACRITICS.sub('', text)                       # remove diacritics
    text = re.sub(r'[إأآا]', 'ا', text)                           # unify alef forms
    text = re.sub(r'ى', 'ي', text)                                # unify yaa
    text = re.sub(r'ة', 'ه', text)                                # unify teh marbuta
    text = re.sub(r'ـ+', '', text)                                # remove tatweel
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)                   # collapse elongation
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)             # keep Arabic chars only
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_arabic(text):
    text = normalize_arabic(text)
    sentences = re.split(r'(?<=[.!؟])\s+', text)
    tokenized = [s.split() for s in sentences if s.strip()]
    return [s for s in tokenized if len(s) >= 3]

en_sentences = preprocess_english(en_text)
ar_sentences = preprocess_arabic(ar_text)

print(f"English: {len(en_sentences)} sentences, {sum(len(s) for s in en_sentences)} tokens, "
      f"vocab≈{len(set(w for s in en_sentences for w in s))}")
print(f"Arabic : {len(ar_sentences)} sentences, {sum(len(s) for s in ar_sentences)} tokens, "
      f"vocab≈{len(set(w for s in ar_sentences for w in s))}")
print("\nSample EN:", en_sentences[0])
print("Sample AR:", ar_sentences[0])


English: 4144 sentences, 91528 tokens, vocab≈12081
Arabic : 5 sentences, 100514 tokens, vocab≈24077

Sample EN: ['anarchism', 'is', 'a', 'political', 'philosophy', 'and', 'movement', 'that', 'is', 'skeptical', 'of', 'all', 'justifications', 'for', 'authority', 'and', 'seeks', 'to', 'abolish', 'the', 'institutions', 'it', 'claims', 'maintain', 'unnecessary', 'coercion', 'and', 'hierarchy', 'typically', 'including', 'and', 'capitalism']
Sample AR: ['الماء', 'ماده', 'شفافه', 'عديمه', 'اللون', 'والرائحه،', 'وهو', 'المكون', 'الاساسي', 'للجداول', 'والبحيرات', 'والبحار', 'والمحيطات', 'وكذلك', 'للسوائل', 'في', 'جميع', 'الكائنات', 'الحيه،', 'وهو', 'اكثر', 'المركبات', 'الكيميائيه', 'انتشارا', 'علي', 'سطح', 'الارض', 'يتالف', 'جزيء', 'الماء', 'من', 'ذره', 'اكسجين', 'مركزيه', 'ترتبط', 'بها', 'ذرتا', 'هيدروجين', 'علي', 'طرفيها', 'برابطه', 'تساهميه', 'بحيث', 'تكون', 'صيغته', 'الكيميائيه', 'عند', 'الظروف', 'القياسيه', 'من', 'الضغط', 'ودرجه', 'الحراره', 'يكون', 'الماء', 'سائلا؛', 'اما', 'الحاله', 'الصل

In [5]:
# Train/test split (sentence-level) — used later for perplexity comparison
def split_sentences(sentences, test_ratio=0.15):
    idx = list(range(len(sentences)))
    random.shuffle(idx)
    n_test = max(1, int(len(idx) * test_ratio))
    test_idx = set(idx[:n_test])
    train = [s for i, s in enumerate(sentences) if i not in test_idx]
    test = [s for i, s in enumerate(sentences) if i in test_idx]
    return train, test

en_train, en_test = split_sentences(en_sentences)
ar_train, ar_test = split_sentences(ar_sentences)
print(f"EN train/test: {len(en_train)}/{len(en_test)}")
print(f"AR train/test: {len(ar_train)}/{len(ar_test)}")


EN train/test: 3523/621
AR train/test: 4/1


## 3. Approach A — N-gram Statistical Language Model

A trigram model with bigram/unigram **stupid backoff** and Laplace-smoothed probability estimates.
Fast to train, fully interpretable, no GPU needed — but limited context window and no generalization
to unseen word combinations.


In [6]:
class NGramLanguageModel:
    """Trigram language model with stupid backoff to bigram/unigram counts."""

    def __init__(self, n=3):
        self.n = n
        self.ngram_counts = [defaultdict(Counter) for _ in range(n)]  # index 0=unigram context,...
        self.vocab = set()

    def fit(self, sentences):
        for sent in sentences:
            tokens = ["<s>"] * (self.n - 1) + sent + ["</s>"]
            self.vocab.update(tokens)
            for order in range(1, self.n + 1):
                for i in range(order - 1, len(tokens)):
                    context = tuple(tokens[i - order + 1:i])
                    word = tokens[i]
                    self.ngram_counts[order - 1][context][word] += 1
        return self

    def predict_next(self, context_text, top_k=5):
        tokens = context_text.split()
        candidates = Counter()
        # Try highest order first (stupid backoff), then fall back
        for order in range(self.n, 0, -1):
            ctx = tuple((["<s>"] * (order - 1) + tokens)[-(order - 1):]) if order > 1 else tuple()
            counts = self.ngram_counts[order - 1].get(ctx)
            if counts:
                total = sum(counts.values())
                vocab_size = max(len(self.vocab), 1)
                for word, c in counts.items():
                    if word not in ("<s>", "</s>"):
                        # Laplace-smoothed probability, discounted at lower orders (stupid backoff weight 0.4)
                        prob = (c + 1) / (total + vocab_size)
                        weight = 1.0 if order == self.n else 0.4 ** (self.n - order)
                        candidates[word] = max(candidates[word], prob * weight)
                if order == self.n and len(candidates) >= top_k:
                    break
        return candidates.most_common(top_k)

    def sentence_logprob(self, sent):
        tokens = ["<s>"] * (self.n - 1) + sent + ["</s>"]
        logprob = 0.0
        vocab_size = max(len(self.vocab), 1)
        for i in range(self.n - 1, len(tokens)):
            ctx = tuple(tokens[i - self.n + 1:i])
            word = tokens[i]
            counts = self.ngram_counts[self.n - 1].get(ctx, Counter())
            total = sum(counts.values())
            prob = (counts.get(word, 0) + 1) / (total + vocab_size)
            logprob += np.log(prob)
        return logprob, len(tokens) - (self.n - 1)

    def perplexity(self, sentences):
        total_logprob, total_tokens = 0.0, 0
        for sent in sentences:
            lp, n_tok = self.sentence_logprob(sent)
            total_logprob += lp
            total_tokens += n_tok
        return float(np.exp(-total_logprob / max(total_tokens, 1)))


ngram_en = NGramLanguageModel(n=3).fit(en_train)
ngram_ar = NGramLanguageModel(n=3).fit(ar_train)
print("N-gram models trained.")
print("EN demo — 'natural language' ->", ngram_en.predict_next("natural language"))
print("AR demo — 'الذكاء الاصطناعي' ->", ngram_ar.predict_next("الذكاء الاصطناعي"))


N-gram models trained.
EN demo — 'natural language' -> [('the', 0.009225366949926261), ('of', 0.0049682461649594185), ('and', 0.0039071764670472455), ('in', 0.0037241780622636026), ('to', 0.0028172122841692336)]
AR demo — 'الذكاء الاصطناعي' -> [('في', 0.004586051943435985), ('من', 0.0033755617050445566), ('علي', 0.0017913630709081218), ('الي', 0.001417654675163117), ('الماء', 0.0008936505115641425)]


## 4. Approach B — LSTM Neural Language Model (Keras)

Trained from scratch on each real corpus: tokenizer → fixed-length context windows → Embedding → LSTM →
Dense(softmax over vocabulary). Training sequences are capped (`MAX_TRAIN_SEQUENCES`) since the real
Wikipedia corpus produces far more windows than a tiny toy corpus — this keeps training time reasonable
on Kaggle while still learning from real text. Increase the cap (and epochs) for higher quality if you
have GPU time to spare.


In [7]:
CONTEXT_LEN = 4              # number of previous tokens used to predict the next one
MAX_TRAIN_SEQUENCES = 80_000  # cap on (context -> next word) training windows per language

def build_lstm_dataset(sentences, context_len=CONTEXT_LEN, max_sequences=MAX_TRAIN_SEQUENCES):
    tokenizer = Tokenizer(oov_token="<OOV>")
    tokenizer.fit_on_texts([" ".join(s) for s in sentences])
    vocab_size = len(tokenizer.word_index) + 1

    sequences = []
    for sent in sentences:
        encoded = tokenizer.texts_to_sequences([" ".join(sent)])[0]
        for i in range(context_len, len(encoded)):
            sequences.append(encoded[i - context_len:i + 1])

    sequences = np.array(sequences)
    if len(sequences) > max_sequences:
        idx = np.random.choice(len(sequences), max_sequences, replace=False)
        sequences = sequences[idx]

    X, y = sequences[:, :-1], sequences[:, -1]
    y = to_categorical(y, num_classes=vocab_size)
    return tokenizer, vocab_size, X, y


def build_lstm_model(vocab_size, context_len=CONTEXT_LEN, embed_dim=64, lstm_units=128):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=context_len),
        LSTM(lstm_units, return_sequences=False),
        Dropout(0.2),
        Dense(vocab_size, activation="softmax"),
    ])
    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model


def train_lstm(sentences, label, epochs=15, batch_size=256):
    tokenizer, vocab_size, X, y = build_lstm_dataset(sentences)
    print(f"[{label}] vocab_size={vocab_size}, training_sequences={len(X)}")
    model = build_lstm_model(vocab_size)
    es = EarlyStopping(monitor="loss", patience=3, restore_best_weights=True)
    model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=1, callbacks=[es])
    print(f"[{label}] LSTM training complete.")
    return model, tokenizer, vocab_size


In [8]:
lstm_model_en, tok_en, vocab_size_en = train_lstm(en_train, "English")


[English] vocab_size=11163, training_sequences=63894
Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 59ms/step - accuracy: 0.0700 - loss: 7.5903
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 58ms/step - accuracy: 0.0734 - loss: 7.0174
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 58ms/step - accuracy: 0.0918 - loss: 6.8572
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 56ms/step - accuracy: 0.1019 - loss: 6.7089
Epoch 5/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 59ms/step - accuracy: 0.1111 - loss: 6.5785
Epoch 6/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 59ms/step - accuracy: 0.1150 - loss: 6.4669
Epoch 7/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 58ms/step - accuracy: 0.1201 - loss: 6.3592
Epoch 8/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 59ms/step - accuracy: 0.1235 - loss: 6.2453
Epoch 9/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 58ms/step - accuracy: 0.1289 - loss: 6.1275
Epoch 10/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 58ms/step - accuracy: 0.1346 - loss: 5.9968
Epoch 11/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 57ms/step - accuracy:

In [9]:
lstm_model_ar, tok_ar, vocab_size_ar = train_lstm(ar_train, "Arabic")


[Arabic] vocab_size=10006, training_sequences=29355
Epoch 1/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 8s 55ms/step - accuracy: 0.0360 - loss: 8.6923
Epoch 2/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.0385 - loss: 8.0637
Epoch 3/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 58ms/step - accuracy: 0.0384 - loss: 7.9434
Epoch 4/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - accuracy: 0.0384 - loss: 7.8400
Epoch 5/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.0384 - loss: 7.7391
Epoch 6/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - accuracy: 0.0385 - loss: 7.6499
Epoch 7/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - accuracy: 0.0385 - loss: 7.5800
Epoch 8/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - accuracy: 0.0389 - loss: 7.5219
Epoch 9/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.0383 - loss: 7.4679
Epoch 10/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.0385 - loss: 7.4165
Epoch 11/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - accuracy: 0.0402 - l

In [10]:
def lstm_predict_next(text, model, tokenizer, top_k=5, context_len=CONTEXT_LEN):
    encoded = tokenizer.texts_to_sequences([text])[0]
    encoded = encoded[-context_len:]
    encoded = pad_sequences([encoded], maxlen=context_len, padding="pre")
    probs = model.predict(encoded, verbose=0)[0]
    top_idx = probs.argsort()[-top_k:][::-1]
    idx_to_word = {v: k for k, v in tokenizer.word_index.items()}
    return [(idx_to_word.get(i, "<OOV>"), float(probs[i])) for i in top_idx]

print("EN demo — 'natural language' ->", lstm_predict_next("natural language", lstm_model_en, tok_en))
print("AR demo — 'الذكاء الاصطناعي' ->", lstm_predict_next("الذكاء الاصطناعي", lstm_model_ar, tok_ar))


EN demo — 'natural language' -> [('and', 0.038294482976198196), ('of', 0.022267663851380348), ('which', 0.018832970410585403), ('for', 0.01496038120239973), ("''", 0.014163902960717678)]
AR demo — 'الذكاء الاصطناعي' -> [('في', 0.02041262947022915), ('من', 0.019193943589925766), ('الذي', 0.01478555891662836), ('مثل', 0.014312940649688244), ('الي', 0.014119839295744896)]


## 5. Approach C — Pretrained Transformers (GPT-2 / AraGPT2)

No training required — these models already encode strong next-token distributions from large-scale
pretraining. English uses `distilgpt2` (fast); Arabic uses `aubmindlab/aragpt2-base`.


In [11]:
print("Loading English transformer (distilgpt2)...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("distilgpt2").to(DEVICE).eval()

print("Loading Arabic transformer (aubmindlab/aragpt2-base)...")
aragpt2_tokenizer = AutoTokenizer.from_pretrained("aubmindlab/aragpt2-base")
aragpt2_model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base").to(DEVICE).eval()
print("Transformer models loaded.")


Loading English transformer (distilgpt2)...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/553M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Transformer models loaded.


In [12]:
def transformer_predict_next(text, model, tokenizer, top_k=5):
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, top_k)
    results = []
    for p, tid in zip(top_probs.tolist(), top_ids.tolist()):
        token = tokenizer.decode([tid]).strip()
        if token:
            results.append((token, p))
    return results

print("EN demo — 'natural language' ->",
      transformer_predict_next("natural language", gpt2_model, gpt2_tokenizer))
print("AR demo — 'الذكاء الاصطناعي' ->",
      transformer_predict_next("الذكاء الاصطناعي", aragpt2_model, aragpt2_tokenizer))


EN demo — 'natural language' -> [('.', 0.22492781281471252), (',', 0.12798510491847992), ('and', 0.0615721195936203), ('of', 0.03567846119403839), ('is', 0.030072486028075218)]
AR demo — 'الذكاء الاصطناعي' -> [('هو', 0.0733160600066185), ('في', 0.04154285043478012), ('،', 0.03656189516186714), ('لا', 0.02821565605700016), ('(', 0.028138455003499985)]


## 6. Evaluation & Side-by-Side Comparison

**Perplexity** (lower = better) is computed for the N-gram and LSTM models on held-out test
sentences — directly comparable since both were trained on the same corpus split.
Pretrained transformers aren't perplexity-compared here since they weren't trained on this
specific small corpus (they'd unfairly look unusual on such a tiny domain-specific test set);
their value is in the qualitative top-k predictions below instead.


In [13]:
def lstm_perplexity(model, tokenizer, sentences, context_len=CONTEXT_LEN):
    # Re-encode using the *trained* tokenizer's vocab for a fair comparison
    sequences = []
    for sent in sentences:
        encoded = tokenizer.texts_to_sequences([" ".join(sent)])[0]
        for i in range(context_len, len(encoded)):
            sequences.append(encoded[i - context_len:i + 1])
    if not sequences:
        return float("nan")
    sequences = np.array(sequences)
    Xs, ys = sequences[:, :-1], sequences[:, -1]
    probs = model.predict(Xs, verbose=0)
    token_probs = probs[np.arange(len(ys)), ys]
    token_probs = np.clip(token_probs, 1e-10, 1.0)
    return float(np.exp(-np.mean(np.log(token_probs))))

ppl_table = pd.DataFrame([
    {"Language": "English", "Model": "N-gram (trigram)", "Perplexity": ngram_en.perplexity(en_test)},
    {"Language": "English", "Model": "LSTM",             "Perplexity": lstm_perplexity(lstm_model_en, tok_en, en_test)},
    {"Language": "Arabic",  "Model": "N-gram (trigram)", "Perplexity": ngram_ar.perplexity(ar_test)},
    {"Language": "Arabic",  "Model": "LSTM",             "Perplexity": lstm_perplexity(lstm_model_ar, tok_ar, ar_test)},
])
ppl_table


,Language,Model,Perplexity
0,English,N-gram (trigram),8517.592507
1,English,LSTM,2024.156616
2,Arabic,N-gram (trigram),9842.191904
3,Arabic,LSTM,35657.582031


In [14]:
def compare_predictions(text, language="en", top_k=5):
    """Show top-k next-word predictions from all three approaches side by side."""
    if language == "en":
        ngram_res = ngram_en.predict_next(text, top_k)
        lstm_res = lstm_predict_next(text, lstm_model_en, tok_en, top_k)
        trans_res = transformer_predict_next(text, gpt2_model, gpt2_tokenizer, top_k)
    else:
        ngram_res = ngram_ar.predict_next(text, top_k)
        lstm_res = lstm_predict_next(text, lstm_model_ar, tok_ar, top_k)
        trans_res = transformer_predict_next(text, aragpt2_model, aragpt2_tokenizer, top_k)

    rows = []
    for rank in range(top_k):
        rows.append({
            "Rank": rank + 1,
            "N-gram": f"{ngram_res[rank][0]} ({ngram_res[rank][1]:.3f})" if rank < len(ngram_res) else "",
            "LSTM": f"{lstm_res[rank][0]} ({lstm_res[rank][1]:.3f})" if rank < len(lstm_res) else "",
            "Transformer": f"{trans_res[rank][0]} ({trans_res[rank][1]:.3f})" if rank < len(trans_res) else "",
        })
    return pd.DataFrame(rows)

print("Prompt: 'natural language'  (English)")
display(compare_predictions("natural language", language="en"))

print("\nPrompt: 'الذكاء الاصطناعي'  (Arabic)")
display(compare_predictions("الذكاء الاصطناعي", language="ar"))


Prompt: 'natural language'  (English)


,Rank,N-gram,LSTM,Transformer
0,1,the (0.009),and (0.038),. (0.225)
1,2,of (0.005),of (0.022),", (0.128)"
2,3,and (0.004),which (0.019),and (0.062)
3,4,in (0.004),for (0.015),of (0.036)
4,5,to (0.003),'' (0.014),is (0.030)



Prompt: 'الذكاء الاصطناعي'  (Arabic)


,Rank,N-gram,LSTM,Transformer
0,1,في (0.005),في (0.020),هو (0.073)
1,2,من (0.003),من (0.019),في (0.042)
2,3,علي (0.002),الذي (0.015),، (0.037)
3,4,الي (0.001),مثل (0.014),لا (0.028)
4,5,الماء (0.001),الي (0.014),( (0.028)


In [15]:
# Latency comparison (single-prediction wall-clock time, averaged over a few runs)
def time_predict(fn, *args, runs=5):
    t0 = time.time()
    for _ in range(runs):
        fn(*args)
    return (time.time() - t0) / runs * 1000  # ms

latency_table = pd.DataFrame([
    {"Model": "N-gram",     "Language": "EN", "Latency (ms)": time_predict(ngram_en.predict_next, "natural language")},
    {"Model": "LSTM",       "Language": "EN", "Latency (ms)": time_predict(lstm_predict_next, "natural language", lstm_model_en, tok_en)},
    {"Model": "Transformer","Language": "EN", "Latency (ms)": time_predict(transformer_predict_next, "natural language", gpt2_model, gpt2_tokenizer)},
    {"Model": "N-gram",     "Language": "AR", "Latency (ms)": time_predict(ngram_ar.predict_next, "الذكاء الاصطناعي")},
    {"Model": "LSTM",       "Language": "AR", "Latency (ms)": time_predict(lstm_predict_next, "الذكاء الاصطناعي", lstm_model_ar, tok_ar)},
    {"Model": "Transformer","Language": "AR", "Latency (ms)": time_predict(transformer_predict_next, "الذكاء الاصطناعي", aragpt2_model, aragpt2_tokenizer)},
])
latency_table


,Model,Language,Latency (ms)
0,N-gram,EN,11.356783
1,LSTM,EN,69.335985
2,Transformer,EN,47.264194
3,N-gram,AR,7.049513
4,LSTM,AR,111.052513
5,Transformer,AR,109.224081


## 7. Autocomplete Demo

A unified `autocomplete()` function plus example calls (non-blocking — safe for Run All).
An optional interactive text box is provided at the end for live typing in a running session.


In [16]:
def autocomplete(text, language="en", model="ensemble", top_k=3):
    """
    model: 'ngram' | 'lstm' | 'transformer' | 'ensemble'
    'ensemble' averages rank position across all three approaches' top suggestions.
    """
    if model == "ensemble":
        df = compare_predictions(text, language, top_k=5)
        votes = Counter()
        for col in ["N-gram", "LSTM", "Transformer"]:
            for rank, val in enumerate(df[col]):
                if val:
                    word = val.split(" (")[0]
                    votes[word] += (5 - rank)
        return [w for w, _ in votes.most_common(top_k)]
    elif model == "ngram":
        res = ngram_en.predict_next(text, top_k) if language == "en" else ngram_ar.predict_next(text, top_k)
    elif model == "lstm":
        res = (lstm_predict_next(text, lstm_model_en, tok_en, top_k) if language == "en"
               else lstm_predict_next(text, lstm_model_ar, tok_ar, top_k))
    elif model == "transformer":
        res = (transformer_predict_next(text, gpt2_model, gpt2_tokenizer, top_k) if language == "en"
               else transformer_predict_next(text, aragpt2_model, aragpt2_tokenizer, top_k))
    else:
        raise ValueError("Unknown model: " + model)
    return [w for w, _ in res]

# Example, non-interactive demo calls
demo_prompts = [
    ("deep learning has", "en"),
    ("machine learning models", "en"),
    ("معالجة اللغة الطبيعية", "ar"),
    ("الشبكات العصبية", "ar"),
]
for prompt, lang in demo_prompts:
    print(f"[{lang}] '{prompt}' -> {autocomplete(prompt, language=lang, model='ensemble')}")


[en] 'deep learning has' -> ['been', 'the', 'a']
[en] 'machine learning models' -> ['of', 'the', 'historical']
[ar] 'معالجة اللغة الطبيعية' -> ['في', 'من', 'الي']
[ar] 'الشبكات العصبية' -> ['في', 'من', 'الي']


In [17]:
# Optional: live interactive widget (only runs meaningfully in an active Kaggle session, won't block Run All)
try:
    import ipywidgets as widgets
    from IPython.display import display as ip_display, clear_output

    lang_dd = widgets.Dropdown(options=[("English", "en"), ("Arabic", "ar")], description="Language:")
    text_box = widgets.Text(description="Type:", placeholder="Start typing...")
    out = widgets.Output()

    def on_change(change):
        with out:
            clear_output()
            if text_box.value.strip():
                suggestions = autocomplete(text_box.value, language=lang_dd.value, model="ensemble", top_k=5)
                print("Suggestions:", suggestions)

    text_box.observe(on_change, names="value")
    lang_dd.observe(on_change, names="value")
    ip_display(widgets.VBox([lang_dd, text_box, out]))
except ImportError:
    print("ipywidgets not available in this environment — skipping interactive widget (demo calls above still work).")


## 8. Save Artifacts

In [18]:
SAVE_DIR = "/kaggle/working/next_word_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# N-gram models
with open(f"{SAVE_DIR}/ngram_en.pkl", "wb") as f:
    pickle.dump(ngram_en, f)
with open(f"{SAVE_DIR}/ngram_ar.pkl", "wb") as f:
    pickle.dump(ngram_ar, f)

# LSTM models + tokenizers
lstm_model_en.save(f"{SAVE_DIR}/lstm_en.keras")
lstm_model_ar.save(f"{SAVE_DIR}/lstm_ar.keras")
with open(f"{SAVE_DIR}/tokenizer_en.json", "w", encoding="utf-8") as f:
    f.write(tok_en.to_json())
with open(f"{SAVE_DIR}/tokenizer_ar.json", "w", encoding="utf-8") as f:
    f.write(tok_ar.to_json())

print("Saved artifacts to:", SAVE_DIR)
print(os.listdir(SAVE_DIR))
# Note: pretrained transformers (gpt2 / aragpt2) are not re-saved here — they're re-downloaded
# from Hugging Face on demand via from_pretrained(), which is the standard approach.


Saved artifacts to: /kaggle/working/next_word_models
['lstm_ar.keras', 'tokenizer_en.json', 'ngram_en.pkl', 'ngram_ar.pkl', 'lstm_en.keras', 'tokenizer_ar.json']


## 9. Summary & Trade-offs

| Approach | Training needed | Data required | Speed | Strength | Weakness |
|---|---|---|---|---|---|
| **N-gram** | Seconds–minutes | Works fine on real data as-is | Fastest | Interpretable, no GPU | Fixed context window, no generalization to unseen phrasing |
| **LSTM** | Minutes (GPU helps) | Trained on real Wikipedia text, but capped to `MAX_TRAIN_SEQUENCES` windows for speed | Fast inference | Learns patterns beyond fixed n-grams | Quality scales with `N_ARTICLES`/`MAX_TRAIN_SEQUENCES`/epochs — raise these for a stronger model |
| **Transformer (pretrained)** | None (zero-shot) | None — already pretrained on massive corpora | Slower per call, but highest quality | Best fluency & world knowledge out of the box | Heavier (more memory/compute), less domain-specific unless fine-tuned |

**Next steps for production:**
- Increase `N_ARTICLES` / `MAX_CHARS` (Section 1) and `MAX_TRAIN_SEQUENCES` / epochs (Section 4) for a noticeably stronger N-gram and LSTM model — current defaults are tuned for a reasonable Kaggle session runtime, not maximum quality.
- Swap the Wikipedia stream for a domain-specific real corpus (set `EN_CORPUS_PATH` / `AR_CORPUS_PATH`) if you want completions tuned to a specific domain (e.g., your agent course notes or Arabic sentiment corpus).
- For the transformer approach, consider fine-tuning `distilgpt2` / `aragpt2-base` on that same domain text for sharper, domain-specific completions.
- For deployment, the N-gram and LSTM artifacts saved in Section 8 are lightweight and easy to serve; the transformer can be served via a Hugging Face `pipeline` behind an API.
